# N044 · 旋转数组、矩阵与峰值

**目标：** 根据局部结构构造正确的舍弃条件。

**先修：** N018, N042。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** 旋转有序；重复导致退化；矩阵单调；峰值存在性；二维转索引。

**配套讲解来源：** [同名逐章意见](../../comment/044_binary_search_structured_data.md)。

本章阅读顺序：**先明白问题 → 补基础 → 手算 → 看核心片段 → 分段运行 → 核对测试**。需要复制使用时，跳到末尾“完整代码”，它包含全部依赖、实现和测试。

## 1. 通俗解释：输入什么，要得到什么

上一章（N042）我们会在"完全有序"的数组上二分。这一章处理三种"不那么有序、但局部仍有秩序"的数据，核心问题都是同一句话：**怎么根据局部结构，构造出正确的"舍弃条件"，让二分还能继续用**。三个具体问题：

1. **旋转数组找数**：一个原本排好序的数组被"旋转"了，比如 `[0,1,2,3,4]` 旋转后变成 `[3,4,0,1,2]`（把前面一段搬到后面）。要在这种数组里找 `target`，返回下标（找不到返回 -1）。有重复值时还要返回 True/False。
2. **矩阵找数**：一个每行有序、且下一行开头比上一行结尾大的矩阵（整体看就是一个"扭成一维"的有序序列），判断 `target` 在不在里面。
3. **找峰值**：数组里相邻元素互不相等，找出任意一个"比左右邻居都大"的位置（下标）。注意题目只要"任意一个峰"，不需要最大的那个。

给一个具体到数字的小例子（旋转数组）：

- 输入：`nums = [3,4,0,1,2]`，`target = 1`。
- 输出：`3`。
- 为什么是 3：数组确实无序（3 > 0），不能直接普通二分；但下标 3 处的值就是 1，我们的算法要在 4 次以内的比较里找到它，而不是从头扫到尾。

## 2. 基础知识：先读懂这些词

- **旋转有序**：把一个有序数组从某个位置切开、前后两段交换，得到的就是"旋转有序数组"。它的特点是：从任何位置看，**至少有一半是有序的**——断点（比如 `[3,4 | 0,1,2]` 里的 4 到 0）只会落在某一半里，另一半完好无损。
- **有序半区判方向**：二分时先看 `nums[lo]` 和 `nums[mid]` 的大小关系，判断哪半边是有序的；再看 `target` 是否落在有序那半边的取值范围内：在，就往那半边走；不在，就走另一半。这就是 “学习目标”部分 说的"根据局部结构构造正确的舍弃条件"。
- **重复导致退化**：如果数组里有重复值，可能出现 `nums[lo] == nums[mid] == nums[hi]`（比如 `[1,0,1,1,1]` 的 lo、mid、hi 都是 1），这时你**完全无法判断哪半边有序**——左半 `[1,0,1]` 看起来"首尾相等"，比较失去信息。唯一的办法是把两端各往里缩一步（`lo+=1; hi-=1`），一轮只去掉两个元素，所以最坏情况退化成 O(n) 的线性扫描。
- **矩阵单调 / 二维转一维索引**：`search_matrix` 处理的矩阵满足"逐行看有序，且下一行第一个数大于上一行最后一个数"，所以把矩阵按行拼接成一个长一维数组后整体有序。技巧是**索引换算**：一维编号 `mid` 对应矩阵里的 `matrix[mid//cols][mid%cols]`（第几行除一下、第几列模一下），不需要真的把矩阵摊平。
- **峰值存在性与"边界视为负无穷"**：约定数组两端的外侧当作无穷小（`nums[-1]` 和 `nums[n]` 视为 -∞）。在这个约定下，相邻不相等的数组**一定存在峰值**：从左往右走，只要一直上升，最右端本身就是峰（右边是 -∞）；中途只要有一次下降，下降点就是峰。这个存在性是二分找峰的底气。
- **爬坡式二分（find_peak）**：看 `nums[mid]` 和 `nums[mid+1]`：如果还在上坡，右边必有峰，去右半；否则 mid 本身或左边有峰，收左边界。

## 3. 思路推导：从小例子开始

### 旋转数组（无重复版）的思路

- **Step 1**：先按普通二分取 `mid`，如果恰好等于 `target` 就直接返回。
- **Step 2**：判断哪半边有序。因为断点只有一处，`nums[lo] <= nums[mid]` 时左半 `[lo..mid]` 一定是升序的（相等也允许，说明没有断点在左半内部）。
- **Step 3**：判断 `target` 在不在有序那半边的"值域"里。左半有序就看 `nums[lo] <= target < nums[mid]`；在就去左半（`hi=mid-1`），不在就去右半（`lo=mid+1`）。右半有序时对称：`nums[mid] < target <= nums[hi]`。
- **Step 4**：重复直到找到或区间为空，返回 -1。

**手算演示：** `nums=[3,4,0,1,2]`，`target=1`。

- 第一轮：`lo=0, hi=4, mid=2`，`nums[2]=0 ≠ 1`。`nums[lo]=3 <= nums[mid]=0`？不成立（3 > 0），说明断点在左半、**右半有序**。检查 `0 < 1 <= nums[hi]=2`？成立，target 在右半的值域里 → `lo=3`。
- 第二轮：`lo=3, hi=4, mid=3`，`nums[3]=1 == target`，返回 3。两次比较就命中了。

**对照 “运行示例”部分 的小实例表格：** notebook 用 `base=[0,1,2,3,4]` 生成 5 种旋转量，对每种数组找目标 2，表格数据是：

| 旋转量 | 数组 | 目标 2 的位置 |
|---|---|---|
| 0 | [0,1,2,3,4] | 2 |
| 1 | [1,2,3,4,0] | 1 |
| 2 | [2,3,4,0,1] | 0 |
| 3 | [3,4,0,1,2] | 4 |
| 4 | [4,0,1,2,3] | 3 |

你可以拿旋转量 3 那行 `[3,4,0,1,2]` 自测：目标 2 在下标 4，我们的手算演示找的是目标 1（下标 3），两个都试一遍，体会"判哪半有序 → 判 target 值域"这套动作。

### 矩阵二分的思路

- **Step 1**：把矩阵当成长度为 `rows*cols` 的一维有序数组，二分这个"虚拟数组"。
- **Step 2**：每次把一维的 `mid` 换算回二维 `matrix[mid//cols][mid%cols]` 取值比较。
- **Step 3**：循环结束后 `lo` 是"第一个 ≥ target 的位置"，最后检查这个位置的值是否真的等于 target（防止 target 比所有数都大时越界误判）。

**手算演示：** `matrix=[[1,3,5],[7,9,11]]`，`target=9`。`rows=2, cols=3`，虚拟长度 6，`lo=0, hi=6`。

- `mid=3` → `matrix[3//3][3%3] = matrix[1][0] = 7 < 9` → `lo=4`。
- `mid=5` → `matrix[1][2] = 11 ≥ 9` → `hi=5`。
- `mid=4` → `matrix[1][1] = 9 ≥ 9` → `hi=4`。
- `lo == hi == 4`，检查 `matrix[1][1] == 9` 成立 → 返回 True。

### 找峰值的思路

- **Step 1**：取 `mid`，比较 `nums[mid]` 和 `nums[mid+1]`。
- **Step 2**：如果 `nums[mid] < nums[mid+1]`（上坡），峰一定在右边（哪怕一路升到头，最右端也是峰），`lo=mid+1`。
- **Step 3**：否则（下坡或已经到顶），mid 或它左边必有峰，`hi=mid`。
- **Step 4**：`lo==hi` 时返回，这个位置就是峰。

**手算演示：** `nums=[1,3,2,0,5]`。

- `lo=0,hi=4,mid=2`：`nums[2]=2 < nums[3]=0`？否 → `hi=2`。
- `lo=0,hi=2,mid=1`：`nums[1]=3 < nums[2]=2`？否 → `hi=1`。
- `lo=0,hi=1,mid=0`：`nums[0]=1 < nums[1]=3`？是 → `lo=1`。
- 返回 1。验证：`nums[1]=3` 大于左邻 1 和右邻 2，确实是峰。（数组里 5 也是峰，但我们只承诺返回"任意一个"。）

## 4. 核心代码：search_rotated_unique

先关注决定答案的状态与更新操作。这是完整实现的对应片段，不是另一套解法；导入、辅助结构及其余接口在下一节补全。

```python
def search_rotated_unique(nums, target):
    lo, hi = (0, len(nums) - 1)
    while lo <= hi:
        mid = (lo + hi) // 2
        if nums[mid] == target:
            return mid
        if nums[lo] <= nums[mid]:
            if nums[lo] <= target < nums[mid]:
                hi = mid - 1
            else:
                lo = mid + 1
        elif nums[mid] < target <= nums[hi]:
            lo = mid + 1
        else:
            hi = mid - 1
    return -1
```

### 逐段读懂代码

### 1. `search_rotated_unique` —— 无重复旋转数组找下标

```python
def search_rotated_unique(nums, target):
    lo, hi = (0, len(nums) - 1)
    while lo <= hi:
        mid = (lo + hi) // 2
        if nums[mid] == target:
            return mid
        if nums[lo] <= nums[mid]:
            if nums[lo] <= target < nums[mid]:
                hi = mid - 1
            else:
                lo = mid + 1
        elif nums[mid] < target <= nums[hi]:
            lo = mid + 1
        else:
            hi = mid - 1
    return -1
```

- `while lo<=hi`：闭区间 `[lo,hi]` 里还有元素就继续；区间空了说明没有 target，返回 -1。
- `if nums[lo]<=nums[mid]:`：左半有序的判定。注意等号：`lo==mid`（区间只剩一两个元素）时也成立，左半单元素当然"有序"。
- `if nums[lo]<=target<nums[mid]:`：Python 的链式比较，一次判断 target 是否严格落在左半的值域 `[nums[lo], nums[mid])` 内。左闭右开是因为 `nums[mid]==target` 已经在上一行被处理掉了。
- `else:` 分支是右半有序的情形，判定 `nums[mid]<target<=nums[hi]`，道理对称，右半值域是 `(nums[mid], nums[hi]]`。
- 每轮要么返回、要么把区间严格砍半，所以是 O(log n)。

### 2. `search_rotated_with_duplicates` —— 有重复时返回 True/False

（对应程序见下方分段实现与完整代码。）

- 与无重复版唯一的差别是第 6 行：三端点相等时（如 `[1,0,1,1,1]` 两头和中点都是 1），你无法分辨断点在哪半，什么信息都得不到，只能把两端各丢一个，赌"答案不在那两个格子"。这正是 “思路推导”部分 说的"有重复时端点相等可能无法判断，只能缩小边界"。
- 返回值从下标改成布尔：因为重复时"第一个出现位置"没有意义，题目（LC 81）只问存在性。
- **手算演示：** `nums=[1,0,1,1,1]`，`target=0`。第一轮 `lo=0,hi=4,mid=2`：`nums[2]=1≠0`；`nums[0]==nums[2]==nums[4]`（全是 1）→ 缩边，`lo=1,hi=3`。第二轮 `mid=2`：`nums[2]=1≠0`；`nums[1]=0 <= nums[2]=1` → 左半有序；`0<=0<1` 成立 → `hi=1`。第三轮 `lo=hi=1,mid=1`：`nums[1]=0==target` → 返回 True。
- 最坏情况：比如 `[1,1,1,1,0,1,1]` 这种数组，每一轮三端都相等，只能一次缩两个，复杂度退化到 O(n)。

### 3. `search_matrix` —— 矩阵二分（二维转一维索引）

```python
def search_matrix(matrix, target):
    if not matrix or not matrix[0]:
        return False
    rows, cols = (len(matrix), len(matrix[0]))
    lo, hi = (0, rows * cols)
    while lo < hi:
        mid = (lo + hi) // 2
        value = matrix[mid // cols][mid % cols]
        if value < target:
            lo = mid + 1
        else:
            hi = mid
    return lo < rows * cols and matrix[lo // cols][lo % cols] == target
```

- 第 2 行挡住空矩阵和"行为空"的矩阵这两类边界输入，直接返回 False。
- `lo,hi=0,rows*cols`：在长度为 `rows*cols` 的虚拟一维数组上做"找第一个 ≥ target"的左闭右开二分。
- `matrix[mid//cols][mid%cols]`：核心的索引换算。比如 `cols=3` 时编号 4 对应 `matrix[1][1]`（4 除以 3 商 1 余 1）。这一行让你不用真的把矩阵摊平成列表。
- `value<target: lo=mid+1 else: hi=mid`：标准模板——值太小就右移，否则收缩到 mid（mid 可能恰好是答案）。
- 最后一行：循环结束时 `lo` 是第一个 ≥ target 的虚拟编号。先检查 `lo<rows*cols`（target 比所有元素都大时 lo 会越界），再确认该位置的值真的等于 target，都通过才返回 True。

### 4. `find_peak` —— 找峰值

```python
def find_peak(nums):
    if not nums:
        raise ValueError('requires a nonempty sequence without equal neighbors')
    lo, hi = (0, len(nums) - 1)
    while lo < hi:
        mid = (lo + hi) // 2
        if nums[mid] < nums[mid + 1]:
            lo = mid + 1
        else:
            hi = mid
    return lo
```

- 第 2 行挡住空数组：没有元素就没有峰，属于调用方违约，直接抛异常（错误信息同时提醒了"相邻元素不能相等"这个前提）。
- `mid=(lo+hi)//2` 且循环条件 `lo<hi` 保证了 `mid+1` 最大是 `hi`，永远不会越界访问。
- `nums[mid]<nums[mid+1]`：还在上坡，峰在右边且**不含 mid**（mid 比右邻矮，不可能是峰），所以 `lo=mid+1`。
- `else: hi=mid`：mid 比右邻高或相等（前提说不会相等，所以是"高"），mid 本身可能就是峰，不能扔，收 `hi=mid`。
- 结束时 `lo==hi`，指向一个峰。

## 5. 分段实现：按顺序运行

**本章接口：** `search_rotated_unique(nums, target)`、`search_rotated_with_duplicates(nums, target)`、`search_matrix(matrix, target)`、`find_peak(nums)`

### search_rotated_unique

In [1]:
def search_rotated_unique(nums, target):
    lo, hi = (0, len(nums) - 1)
    while lo <= hi:
        mid = (lo + hi) // 2
        if nums[mid] == target:
            return mid
        if nums[lo] <= nums[mid]:
            if nums[lo] <= target < nums[mid]:
                hi = mid - 1
            else:
                lo = mid + 1
        elif nums[mid] < target <= nums[hi]:
            lo = mid + 1
        else:
            hi = mid - 1
    return -1

### search_rotated_with_duplicates

In [2]:
def search_rotated_with_duplicates(nums, target):
    lo, hi = (0, len(nums) - 1)
    while lo <= hi:
        mid = (lo + hi) // 2
        if nums[mid] == target:
            return True
        if nums[lo] == nums[mid] == nums[hi]:
            lo += 1
            hi -= 1
        elif nums[lo] <= nums[mid]:
            if nums[lo] <= target < nums[mid]:
                hi = mid - 1
            else:
                lo = mid + 1
        elif nums[mid] < target <= nums[hi]:
            lo = mid + 1
        else:
            hi = mid - 1
    return False

### search_matrix

In [3]:
def search_matrix(matrix, target):
    if not matrix or not matrix[0]:
        return False
    rows, cols = (len(matrix), len(matrix[0]))
    lo, hi = (0, rows * cols)
    while lo < hi:
        mid = (lo + hi) // 2
        value = matrix[mid // cols][mid % cols]
        if value < target:
            lo = mid + 1
        else:
            hi = mid
    return lo < rows * cols and matrix[lo // cols][lo % cols] == target

### find_peak

In [4]:
def find_peak(nums):
    if not nums:
        raise ValueError('requires a nonempty sequence without equal neighbors')
    lo, hi = (0, len(nums) - 1)
    while lo < hi:
        mid = (lo + hi) // 2
        if nums[mid] < nums[mid + 1]:
            lo = mid + 1
        else:
            hi = mid
    return lo

## 6. 运行示例与状态可视化

先预测结果再运行。手算表帮助解释过程；下面的表由当前实现计算。若表只展示输入输出，它并不代替内部状态轨迹。

In [5]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

base=[0,1,2,3,4]
show_table(['旋转量','数组','目标2的位置'],[(k,base[k:]+base[:k],search_rotated_unique(base[k:]+base[:k],2)) for k in range(5)])

旋转量,数组,目标2的位置
0,"[0, 1, 2, 3, 4]",2
1,"[1, 2, 3, 4, 0]",1
2,"[2, 3, 4, 0, 1]",0
3,"[3, 4, 0, 1, 2]",4
4,"[4, 0, 1, 2, 3]",3


## 7. 正确性、适用条件与复杂度

无重复旋转数组的一半保有整体次序，区间成员测试可安全排除另一半；重复版本只在确实无法判别时去端点。峰值若中部上升，右半必在某处转降或在右边界形成峰。

**代价：** 无重复搜索、矩阵查找、峰值 O(log n)；重复搜索最坏 O(n)。均O(1)辅助空间。

### 展开理解

**无重复旋转搜索为什么对？** 断点（值突然变小的地方）整个数组里只有一处，所以 `[lo, mid]` 和 `[mid, hi]` 这两半里**至少有一半是完全没有断点的，也就是完全升序的**。对完全升序的半边，"target 在它的值域范围内"这个判断是百分之百可靠的——在，target 要么在这半边、要么根本不存在于数组（但没关系，我们继续往里走最后会排除掉），不在，target 百分之百不在这半边，可以放心整半扔掉。每轮扔一半，所以是 O(log n)。

**重复版为什么对但会变慢？** 三端点相等时我们得不到任何"哪半有序"的信息，只能缩边，这一步不保证砍半，只是保证不丢答案——被扔掉的两个端点值都等于 `nums[mid]`，而 `nums[mid] ≠ target`，所以那两个位置不可能是答案。如果数组里重复值不多，绝大部分轮次仍走二分分支；只有反复撞上"三端相等"才退化为 O(n)。这是这类题的已知下界——不是写法不行，是重复元素真的能让任何算法都分不清方向。

**矩阵二分为什么对？** 前提是"下一行的第一个数大于上一行的最后一个数"，这保证了把矩阵按行拼起来就是一个严格递增的长数组，虚拟编号 `0..rows*cols-1` 与值的大小顺序完全一致，所以普通二分的所有结论原封不动成立，O(log(rows·cols))。

**找峰为什么对？** 我们的循环始终维持一个承诺："区间 `[lo,hi]` 里一定存在峰"。初始成立（第二节讲过，边界视为负无穷时全数组必有峰）。每轮看中点和它的右邻：如果中点更小，说明此处正在上坡，从 mid+1 到右边界这一段要么中途转降（转降处是峰）、要么一路升到右边界（右边界右侧是负无穷，右边界就是峰），所以右半必有峰，把 lo 移过去后承诺仍成立；如果中点更大，mid 本身或它左边必有峰（同理，往左看要么转降要么落到左边界），hi 收到 mid 承诺也成立。区间缩到只剩一个点时，这个点就是峰。复杂度 O(log n)。

**空间呢？** 四个函数都只用 `lo、hi、mid` 等常数个变量，辅助空间都是 O(1)。

## 8. 单元测试：每个用例在检查什么

“自动测试”部分 的测试全是**暴力对拍**（拿小规模全集和朴素解法比对），这比手挑几个用例可靠得多：

- 第一段 `for n in range(1,7)`：对长度 1 到 6 的每个有序数组 `base=[0..n-1]`，枚举全部 n 种旋转，再对每个 `t`（从 -1 到 n+1，覆盖"比最小还小、在数组中、比最大还大"三类值）断言 `search_rotated_unique` 的返回值与"直接用 `a.index(t)`（不在就 -1）"这个朴素参考完全一致。它测的是正确性 + 找不到时的 -1 约定。
- 第二段嵌套 `combinations_with_replacement([0,1,2],n)`：枚举所有元素取自 {0,1,2} 的非降数组（自带重复），同样枚举所有旋转和 target 0..3，断言重复版返回值等于"t 在不在 a 里"。它专门打"三端相等、无法判向"的退化分支。
- 第三段 `product(range(3),repeat=5)`：枚举所有 3⁵=243 个长度 5 的数组，跳过相邻相等的（违反前提），对每个 `find_peak` 的返回下标 `p` 验证峰的数学定义：`p==0 或 a[p]>a[p-1]`，**且** `p==4 或 a[p]>a[p+1]`。注意它验证的是"是不是峰"，而不是"是不是某个特定下标"，因为题目允许返回任意峰。
- `assert search_matrix([[1,3,5],[7,9,11]],9)`：一个普通正常值用例，9 在矩阵正中间（matrix[1][1]），验证索引换算和收尾判断都对。这条也是第三节手算演示的原型。
- `print('N044: 所有本章断言通过')` 只是打一行确认信息。

In [6]:
from itertools import combinations_with_replacement, product

for n in range(1, 7):
    base = list(range(n))
    for k in range(n):
        a = base[k:] + base[:k]
        for t in range(-1, n + 1):
            assert search_rotated_unique(a, t) == (a.index(t) if t in a else -1)
    for base in combinations_with_replacement([0, 1, 2], n):
        for k in range(n):
            a = base[k:] + base[:k]
            for t in range(4):
                assert search_rotated_with_duplicates(a, t) == (t in a)

for a in product(range(3), repeat=5):
    if all((a[i] != a[i + 1] for i in range(4))):
        p = find_peak(a)
        assert (p == 0 or a[p] > a[p - 1]) and (p == 4 or a[p] > a[p + 1])

assert search_matrix([[1, 3, 5], [7, 9, 11]], 9)

print('N044: 所有本章断言通过')

N044: 所有本章断言通过


## 9. 练习：先尝试，再看提示

**练习 1：** 处理旋转数组重复值。

**练习 2：** 比较74的全局有序与240的行列有序。

<details>
<summary>展开思路提示（不是完整答案）</summary>

- **练习 1（处理旋转数组重复值）**：方向是亲手构造"逼出退化分支"的例子。建议输入取 `[1,1,1,0,1]` 找 0，先手算哪一轮会碰到 `nums[lo]==nums[mid]==nums[hi]`、缩边后区间变成什么；再想清楚缩边为什么不会把答案丢掉（被丢掉的端点值等于 mid 的值，而 mid 不等于 target）。验证时可以用 `t in a` 这种一行暴力解对拍。
- **练习 2（比较 74 与 240）**：74 是本章实现的"整体有序"矩阵（下一行首元素大于上一行尾元素），可以整体二分；240（Search a 2D Matrix II）只有"每行有序、每列有序"，行与行之间没有这个衔接关系，整体二分不成立。建议你写一个 240 型的小矩阵（比如 `[[1,4,7],[2,5,8],[3,6,9]]`），把它按行摊平成 `[1,4,7,2,5,8,3,6,9]`，亲眼看到摊平后并不有序，这就证明了 74 的方法不能迁移；240 的正确做法是从右上角（或左下角）出发走阶梯排除，每次比较排除一行或一列。

</details>

## 10. 完整代码：可独立运行

下面这一格包含**全部导入、全部接口和验证用例**，不依赖前面的单元格。可以重启内核后只运行这一格，也可以复制到 `.py` 文件后运行。在 Notebook 中 Run All 时会再验证一次，这是为独立复制使用保留的完整版本。

In [7]:
# 完整实现：本单元格不依赖其他单元格。
def search_rotated_unique(nums, target):
    lo, hi = (0, len(nums) - 1)
    while lo <= hi:
        mid = (lo + hi) // 2
        if nums[mid] == target:
            return mid
        if nums[lo] <= nums[mid]:
            if nums[lo] <= target < nums[mid]:
                hi = mid - 1
            else:
                lo = mid + 1
        elif nums[mid] < target <= nums[hi]:
            lo = mid + 1
        else:
            hi = mid - 1
    return -1

def search_rotated_with_duplicates(nums, target):
    lo, hi = (0, len(nums) - 1)
    while lo <= hi:
        mid = (lo + hi) // 2
        if nums[mid] == target:
            return True
        if nums[lo] == nums[mid] == nums[hi]:
            lo += 1
            hi -= 1
        elif nums[lo] <= nums[mid]:
            if nums[lo] <= target < nums[mid]:
                hi = mid - 1
            else:
                lo = mid + 1
        elif nums[mid] < target <= nums[hi]:
            lo = mid + 1
        else:
            hi = mid - 1
    return False

def search_matrix(matrix, target):
    if not matrix or not matrix[0]:
        return False
    rows, cols = (len(matrix), len(matrix[0]))
    lo, hi = (0, rows * cols)
    while lo < hi:
        mid = (lo + hi) // 2
        value = matrix[mid // cols][mid % cols]
        if value < target:
            lo = mid + 1
        else:
            hi = mid
    return lo < rows * cols and matrix[lo // cols][lo % cols] == target

def find_peak(nums):
    if not nums:
        raise ValueError('requires a nonempty sequence without equal neighbors')
    lo, hi = (0, len(nums) - 1)
    while lo < hi:
        mid = (lo + hi) // 2
        if nums[mid] < nums[mid + 1]:
            lo = mid + 1
        else:
            hi = mid
    return lo

# 示例与回归测试
from itertools import combinations_with_replacement, product

for n in range(1, 7):
    base = list(range(n))
    for k in range(n):
        a = base[k:] + base[:k]
        for t in range(-1, n + 1):
            assert search_rotated_unique(a, t) == (a.index(t) if t in a else -1)
    for base in combinations_with_replacement([0, 1, 2], n):
        for k in range(n):
            a = base[k:] + base[:k]
            for t in range(4):
                assert search_rotated_with_duplicates(a, t) == (t in a)

for a in product(range(3), repeat=5):
    if all((a[i] != a[i + 1] for i in range(4))):
        p = find_peak(a)
        assert (p == 0 or a[p] > a[p - 1]) and (p == 4 or a[p] > a[p + 1])

assert search_matrix([[1, 3, 5], [7, 9, 11]], 9)

print('N044: 所有本章断言通过')

N044: 所有本章断言通过


## 11. 代表题与迁移

- **33. Search in Rotated Sorted Array**：本章 `search_rotated_unique` 的原型题，练的是"判断哪半有序 + 值域判方向"这套舍弃条件。
- **81. Search in Rotated Sorted Array II**：`search_rotated_with_duplicates` 的原型题，练的是"三端相等时缩边"这个退化处理和"最坏 O(n)"的成因。
- **74. Search a 2D Matrix**：`search_matrix` 的原型题，练的是"二维转一维索引（mid//cols, mid%cols）的整体二分"。
- **162. Find Peak Element**：`find_peak` 的原型题，练的是"上坡往右必有峰"的存在性论证和爬坡式二分。

**题面入口：** 

题目用于知识映射；本章完整代码遵循教学接口，不等于每道题的完整平台提交。

## 12. 掌握检查

合上代码，说明输入输出、核心变量的含义、每次更新为什么成立，以及一个边界情况。再完成一道练习，不能只凭“运行通过”判断掌握。

**继续学习：** [课程索引](../../COURSE_INDEX.md)。